# Experiment 5: Temperature Sensitivity

Grid search over τ_geometry × τ_discovery to understand how temperature
parameters affect training quality and circuit discovery.

- **τ_geometry**: controls softness of contrastive targets during training
- **τ_discovery**: controls sharpening during span-centric clustering (swept post-hoc)

## Colab Setup
Clones the repo, installs dependencies, and mounts Google Drive.

In [ ]:
import os, sys

# 1. Clone repository
REPO_DIR = '/content/trainable-circuits'
if not os.path.exists(REPO_DIR):
    !git clone https://github.com/jacobposchl/trainable-circuits {REPO_DIR}

os.chdir(REPO_DIR)
sys.path.insert(0, REPO_DIR)

# 2. Install extra dependencies
!pip install hdbscan umap-learn -q

# 3. Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

# 4. Path constants
DRIVE_DIR  = '/content/drive/MyDrive/ctls'
DATA_DIR   = DRIVE_DIR + '/data/raw'
CKPT_DIR   = DRIVE_DIR + '/experiments'
CONFIG_DIR = REPO_DIR  + '/configs'

print('Repo:     ', REPO_DIR)
print('Data:     ', DATA_DIR)
print('Checkpts: ', CKPT_DIR)

In [ ]:
import torch
import yaml
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from models.backbone import FrozenBackbone
from models.meta_encoder import MetaEncoder
from data.cifar import get_standard_loaders
from evaluation.circuit_analysis import CircuitAnalyzer
from evaluation.discovery import SpanCentricDiscovery

In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Using device:', device)

# Temperature grids
tau_geometry_values  = [0.05, 0.1, 0.2, 0.5]   # requires separate trained models
tau_discovery_values = [0.1, 0.3, 0.5, 1.0]    # swept post-hoc on one model

In [ ]:
# Load base model (default tau_geometry from phase1.yaml)
config_path     = CONFIG_DIR + '/phase1.yaml'
checkpoint_path = CKPT_DIR  + '/phase1/best.pt'

with open(config_path) as f:
    config = yaml.safe_load(f)
config['data']['data_dir'] = DATA_DIR

backbone = FrozenBackbone(
    arch=config['model']['arch'],
    num_classes=config['model']['num_classes'],
    pretrained=config['model']['pretrained'],
).to(device)

meta_encoder = MetaEncoder(
    layer_dims=backbone.layer_dims,
    projection_dim=config['model']['meta_encoder']['projection_dim'],
    n_heads=config['model']['meta_encoder']['n_heads'],
    n_transformer_layers=config['model']['meta_encoder']['n_transformer_layers'],
    dropout=config['model']['meta_encoder']['dropout']
).to(device)

ckpt = torch.load(checkpoint_path, map_location=device, weights_only=False)
meta_encoder.load_state_dict(ckpt['meta_encoder_state'])
meta_encoder.eval()
print('Model loaded.')

_, val_loader = get_standard_loaders(
    data_dir=DATA_DIR,
    batch_size=config['data'].get('batch_size', 256),
    num_workers=0,
    augment=False,
    download=False,
)

analyzer = CircuitAnalyzer(backbone, meta_encoder, val_loader, device)
data     = analyzer.collect_representations(max_samples=2000)

# Pair profiles [N_pairs, L]
N = data['z_list'][0].shape[0]
idx_a, idx_b = torch.triu_indices(N, N, offset=1)
MAX_PAIRS = 50_000
if idx_a.shape[0] > MAX_PAIRS:
    perm  = torch.randperm(idx_a.shape[0])[:MAX_PAIRS]
    idx_a, idx_b = idx_a[perm], idx_b[perm]

true_sims    = CircuitAnalyzer.compute_pair_profiles(data['trajectories'], idx_a, idx_b)
true_sims_np = true_sims.numpy()
L            = len(data['z_list'])

print(f'N pairs: {idx_a.shape[0]}, L layers: {L}')

In [ ]:
# Sweep tau_discovery (post-hoc, no retraining needed)
n_circuits_tau = []

for tau_d in tau_discovery_values:
    disc = SpanCentricDiscovery(
        n_layers=L,
        tau_discovery=tau_d,
        min_cluster_fraction=config['discovery']['min_cluster_fraction'],
        min_cluster_size=config['discovery']['hdbscan_min_cluster_size'],
    )
    circuits = disc.discover_all(true_sims_np)
    n_circuits_tau.append(len(circuits))
    print(f'\u03c4_discovery = {tau_d}: {len(circuits)} circuits')

fig, ax = plt.subplots(figsize=(6, 4))
ax.plot(tau_discovery_values, n_circuits_tau, 'o-', color='steelblue')
ax.set_xlabel('\u03c4_discovery')
ax.set_ylabel('Number of canonical circuits')
ax.set_title('\u03c4_discovery Sensitivity')
plt.tight_layout()
plt.show()

In [ ]:
# Full tau_geometry x tau_discovery grid heatmap
# Requires training one model per tau_geometry value.
# TODO: Populate once experiments are complete.
#
# tau_geometry_checkpoints = {
#     0.05: CKPT_DIR + '/temp_sensitivity/tau_geom_0.05/best.pt',
#     0.1:  CKPT_DIR + '/phase1/best.pt',
#     0.2:  CKPT_DIR + '/temp_sensitivity/tau_geom_0.2/best.pt',
#     0.5:  CKPT_DIR + '/temp_sensitivity/tau_geom_0.5/best.pt',
# }